# Evaporative Cooling — Quadrupole Trap

The density of states gives polylogarithm orders $\nu = 9/2$ (particle number)
and $\nu = 11/2$ (energy). Energy prefactor $d/2 = 9/2$.

**Reference:** Arvizu-Velázquez et al., arXiv (2026).

In [ ]:
import numpy as np
import mpmath as mp
import math
import scipy.special as ss
from matplotlib import pyplot as plt
import time

from evap_cooling_utils import (
    ConstantsSI, ConstantsEV,
    g_tilde, g_bar,
    newton_raphson_1var, newton_raphson_2var_fused, newton_raphson_2var_fused_real,
    mb_particle_number, mb_temperature_box, mb_temperature_quadrupole, mb_temperature_oscillator,
    create_result_dict, create_mb_result_dict,
    build_cutoff_schedule, initialize_boson_state, initialize_mb_state,
    run_quantum_evaporation, run_mb_evaporation,
    plot_combined_overview, plot_individual_panels,
)

## 1. Physical Parameters

In [ ]:
h = ConstantsEV.h; hb = ConstantsEV.hbar; kB = ConstantsEV.kB; m = ConstantsEV.m_Na23
A = 1e-15; V = 1 / A**3
N0 = 1e7; T0 = 5e-5

def dos_prefactor(T):
    return (64 * kB**6 * m**3 * mp.pi**4 * T**6 * V) / h**6

## 2. Equation of State

In [ ]:
def N_equation_boson(x):
    return dos_prefactor(T0) * mp.polylog(9/2, mp.exp(x)) - N0
def dN_equation_boson(x):
    return dos_prefactor(T0) * mp.polylog(7/2, mp.exp(x))
def N_equation_fermion(x):
    return -dos_prefactor(T0) * mp.polylog(9/2, -mp.exp(x)) - N0
def dN_equation_fermion(x):
    return -dos_prefactor(T0) * mp.polylog(7/2, -mp.exp(x))

## 3. Compute Initial Chemical Potentials

In [ ]:
start = time.time()
alpha_b = newton_raphson_1var(N_equation_boson, dN_equation_boson, -3.8, -3.79, dx=1e-5, precision=50)
if alpha_b is None: raise RuntimeError("Root not found for boson μ")
mu0_b = alpha_b * kB * T0
E0_b = float((9/2) * N0 * kB * T0 * (mp.polylog(11/2, mp.exp(alpha_b)) / mp.polylog(9/2, mp.exp(alpha_b))))
print(f"Bosons:   α = {alpha_b:.10f},  μ = {mu0_b:.6e},  E₀ = {E0_b:.6e}  [{time.time()-start:.1f}s]")

start = time.time()
alpha_f = newton_raphson_1var(N_equation_fermion, dN_equation_fermion, -3.8, -3.79, dx=1e-5, precision=50)
if alpha_f is None: raise RuntimeError("Root not found for fermion μ")
mu0_f = alpha_f * kB * T0
g112 = -mp.polylog(11/2, -mp.exp(alpha_f)); g92 = -mp.polylog(9/2, -mp.exp(alpha_f))
E0_f = float((9/2) * N0 * kB * T0 * (g112 / g92))
print(f"Fermions: α = {alpha_f:.10f},  μ = {mu0_f:.6e},  E₀ = {E0_f:.6e}  [{time.time()-start:.1f}s]")

## 4. Build Cut-off Schedule & Initialize Data

In [ ]:
Q0 = 5e-4; dQ = 1e-6; N_STEPS = 1000
Q_schedule = build_cutoff_schedule(Q0, dQ, N_STEPS)
results_b = create_result_dict(); results_f = create_result_dict(); results_mb = create_mb_result_dict()
initialize_boson_state(results_b, N0, T0, mu0_b, E0_b)
initialize_boson_state(results_f, N0, T0, mu0_f, E0_f)
initialize_mb_state(results_mb, N0, T0)
results_b['Q'] = Q_schedule; results_f['Q'] = Q_schedule; results_mb['Q'] = Q_schedule

## 5. Fused Truncated Distribution Functions

For the quadrupole ($\nu = 9$), the recurrences are:

$$\frac{N_i}{N_{i-1}} = \frac{\tilde{g}_{9/2}}{g_{9/2}}
  - \frac{2}{\sqrt{\pi}}\,\eta^{1/2}\,\frac{\bar{g}_{4}}{g_{9/2}}$$

$$\frac{E_i}{E_{i-1}} = \frac{\tilde{g}_{11/2}}{g_{11/2}}
  - \frac{2}{\sqrt{\pi}}\,\eta^{1/2}\,\frac{\bar{g}_{5}}{g_{11/2}}
  - \frac{4}{9\sqrt{\pi}}\,\eta^{3/2}\,\frac{\bar{g}_{4}}{g_{11/2}}$$

In [ ]:
def _make_truncated_NE_quad(sign):
    def truncated_NE(Ni, Ti, Mui, Ei, Qc):
        eta_c = Qc / Ti
        alpha = Mui / (kB * Ti)
        z_full = sign * mp.exp(alpha)

        gt_92  = g_tilde(9/2, alpha, eta_c, sign)
        gt_112 = g_tilde(11/2, alpha, eta_c, sign)
        g92_full  = mp.polylog(9/2, z_full)
        g112_full = mp.polylog(11/2, z_full)
        gb_4 = g_bar(4, alpha, eta_c, sign)
        gb_5 = g_bar(5, alpha, eta_c, sign)

        g92_norm  = sign * g92_full
        g112_norm = sign * g112_full
        c1 = 2 * mp.sqrt(eta_c / mp.pi)
        c2 = (4 * eta_c**1.5) / (9 * mp.sqrt(mp.pi))

        N1 = (gt_92 / g92_norm - c1 * gb_4 / g92_norm) * Ni
        E1 = (gt_112 / g112_norm - c1 * gb_5 / g112_norm - c2 * gb_4 / g112_norm) * Ei

        return (float(mp.nstr(mp.re(N1), 35) or 0),
                float(mp.nstr(mp.re(E1), 35) or 0))
    return truncated_NE

truncated_NE_boson_quad   = _make_truncated_NE_quad(+1)
truncated_NE_fermion_quad = _make_truncated_NE_quad(-1)

## 6. Newton-Raphson (2-Variable) with Precomputed Polylogs

In [ ]:
def _make_nr_solver_quad(sign):
    def solver(T_init, mu_init, dT, dmu, Ni, Ei):
        def jacobian(T, mu):
            z = sign * mp.exp(mu / (kB * T))
            C = (64 * kB**6 * m**3 * mp.pi**4 * T**6 * V) / h**6
            g72 = mp.polylog(7/2, z); g92 = mp.polylog(9/2, z); g112 = mp.polylog(11/2, z)
            f_val = sign * C * g92 - Ni
            g_val = (9/2) * Ni * kB * T * (g112 / g92) - Ei
            c1 = (64 * kB**5 * m**3 * mp.pi**4 * T**5 * V) / h**6
            f_y_val = sign * c1 * g72
            c2 = (64 * kB**5 * m**3 * mp.pi**4 * T**4 * V) / h**6
            f_x_val = sign * c2 * (-mu * g72 + 6 * kB * T * g92)
            g_y_val = (9 * Ni / 2) * (1 - g72 * g112 / g92**2)
            g_x_val = (9 * Ni / 2) * (-mu / T + mu * g72 * g112 / (T * g92**2) + kB * g112 / g92)
            return (f_val, g_val, f_x_val, f_y_val, g_x_val, g_y_val)
        return newton_raphson_2var_fused_real(jacobian, T_init, mu_init, dT, dmu)
    return solver

nr_solver_boson_quad   = _make_nr_solver_quad(+1)
nr_solver_fermion_quad = _make_nr_solver_quad(-1)

## 7. Run Evaporation Simulations

In [ ]:
dT_nr = 1e-50; dmu_nr = 1e-50

print("Running boson evaporation...")
start = time.time()
run_quantum_evaporation(results_b, truncated_NE_boson_quad, nr_solver_boson_quad, N0, N_STEPS, dT_nr, dmu_nr)
print(f"  Done in {time.time()-start:.1f}s")

print("Running fermion evaporation...")
start = time.time()
run_quantum_evaporation(results_f, truncated_NE_fermion_quad, nr_solver_fermion_quad, N0, N_STEPS, dT_nr, dmu_nr)
print(f"  Done in {time.time()-start:.1f}s")

print("Running Maxwell-Boltzmann evaporation...")
start = time.time()
run_mb_evaporation(results_mb, mb_particle_number, mb_temperature_quadrupole, N0, N_STEPS)
print(f"  Done in {time.time()-start:.1f}s")

## 8. Results

In [ ]:
fig = plot_combined_overview(results_b, results_f, results_mb, "Quadrupole")
plt.show()

In [ ]:
fig = plot_individual_panels(results_b, results_f, results_mb, "Quadrupole")
plt.show()